# Herhalingslabo Classificatie

Hoe weet je wanneer je welk algoritme best gebruikt ? Gebruik volgende handleiding:

**KNearestNeighbours**
- Geschikt voor classificatie en regressie
- Voor kleine/middelgrote datasets
- heel simpel, kan snel worden getrained
- werkt niet goed voor datasets met heel veel features

**Support Vector Machines**
- Vooral een standaard voor binaire classificatie, of wanneer je een duidelijke marge wilt op classificatie (=het hoofddoel van het algoritme)
- Voor kleine/middelgrote datasets
- Is kostelijker om te trainen dan KNN, maar classificeert dan wel sneller
- Kernel keuze en hyperparameter tuning zijn essentieel

**Decision Trees**
- Wanneer je een 'white box', 'explainable' model wilt
- kan zowel numerieke als categorische features makkelijk aan
- kan heel goed om met missende datapunten
- het risico op overfitting is groot. Bij twijfel, gebruik een random forest

**Random forest**
- Zelfde voordelen als decision tree, minder risico op overfitting
- Geschikt voor middelgrote/grote datasets
- Geschikt voor data met (heel) veel features
- Hebben uiteraard meer trainingstijd nodig dan een enkele decision tree

**Adaboost**
- Als data niet goed is gebalanceerd (niet evenveel items van elke klasse)
- Vermijd een onderfit. Dit model is beter dan 1 enkele onderliggende decision tree

**XGBoost**
- XGBoost is bijna *altijd* een goede keuze
- Middelgrote / grote datasets
- Datasets met (heel) veel features
- De implementatie van XGBoost staat, ondanks dat het een boosting algoritme is, toch deels parallellisatie toe.

In de praktijk zijn er vaak meerdere goede keuzes. Het is belangrijk te weten waarom een bepaald model wel/niet een goede keuze zou zijn, en vervolgens de meest veelbelovende modellen te proberen en te kijken wat het beste werkt.

### Wat is een kleine / middelgrote / grote dataset ?

Uiteraard is dat een beetje afhankelijk van hoelang je bereid bent te wachten. In deze cursus gebruiken we de gangbare regel:
- Klein: <10k
- Middelgroot: 10k - 100k
- Groot: >100k

Dit is ongetwijfeld over een paar jaar volledig voorbijgestreefd. Besef dat je in real life dit altijd moet afwegen tegenover de beschikbare compute power, op een Raspberry Pi 3 zal een middelgrote dataset al een stevige trainingstijd hebben, terwijl dat kinderspel is op een cloud compute server.

## Oefeningen

Behandel de volgende 3 datasets - je kan ze downloaden vanop Digitap. Lees in, bekijk, en beoordeel welke algoritmes het meeste zin hebben. Pas de ML algoritmes toe en rapporteer de uiteindelijke score van je model.
- Diabetes - i de 'outcome' kolom wordt aangegeven of mensen diabetes hebben
- Customer Churn: in commerciële setting wordt vaak gesproken van 'churn', oftewel het aangeven in een soort diagram waar klanten in een process afhaken. Deze dataset bevat een 'churn' kolom die dat codeert.
- Creditcard bevat data die reeds door een ander algoritme is gegaan, waardoor de kolommen geen herkenbare naam meer hebben. De laatste kolom wijst op fraude (1) of een valide transactie (0).

In [17]:
# Oefening 1
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Stap 1 - Dataset inladen
diabetes = pd.read_csv("diabetes.csv")

# Stap 2 - Dataset bekijken
print("Eerste en laatste 5 instanties:")
print(diabetes.head())
print(diabetes.tail())

print("\nSamenvatting van de dataset:")
print(diabetes.info())

# Stap 3 - Beoordeling en keuze algoritme
# Het aantal features is beperkt tot 7 en het aantal instanties is ook beperkt tot 767. De outcome-kolom geeft aan dat het hier om een binaire classificatie gaat: iemand heeft op basis van de features diabetes of niet. Omwille van het gegeven dat het hier om een medisch probleem gaat, is een duidelijke marge en een SVM dus gewenst. De beperkte grootte van de dataset laat dit toe.

# Stap 4 - Preprocessing
# Features en labels splitsen
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
X = diabetes[features]
y = diabetes[['Outcome']]
onmogelijke_nullen = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
X[onmogelijke_nullen] = X[onmogelijke_nullen].replace(0, np.nan)
X.fillna(X.median(), inplace=True)

# Train en testset maken
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling van de features is noodzakelijk bij SVM
scaler = StandardScaler()
scaler = scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Stap 5 - Model inladen, GridSearch, trainen en laten voorspellen
svm = SVC(random_state=42)

param_grid = {
    'C': [0.1, 1, 10, 100], 
    'kernel': ['rbf', 'linear']
}

grid = GridSearchCV(svm, param_grid, cv=5, refit=True, scoring='f1')
grid = grid.fit(X_train_scaled, y_train)
y_pred_diabetes = grid.predict(X_test_scaled)

# Stap 6 - Model evalueren en rapporteren
accuracy_diabetes = accuracy_score(y_test, y_pred_diabetes)
confusion_diabetes = confusion_matrix(y_test, y_pred_diabetes)
report_diabetes = classification_report(y_test, y_pred_diabetes)
print(f"\nAccurarcy-score model: {accuracy_diabetes:.4f}")
print(f"\nConfusion-matrix:\n{confusion_diabetes}")
print(f"\nClassificatie-rapport:\n{report_diabetes}")

# Deze score is gezien de medische context niet heel goed, zelfs na een gridsearch. Maar ongeveer 3 op 4 voorspellingen is juist en in het geval van diabetesdiagnoses is niet alleen het aantal vals negatieven (17), maar ook het aantal vals positieven (21) aanzienlijk en een probleem.

Eerste en laatste 5 instanties:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
763           10      101             76             48      180  32.9   
764            2      122             70             27        0  36.8   
765            5      121             72         

/tmp/ipykernel_1041/373770800.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[onmogelijke_nullen] = X[onmogelijke_nullen].replace(0, np.nan)
/tmp/ipykernel_1041/373770800.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.fillna(X.median(), inplace=True)
/opt/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/venv/lib/python3.13/site-packages/sk


Accurarcy-score model: 0.7532

Confusion-matrix:
[[82 17]
 [21 34]]

Classificatie-rapport:
              precision    recall  f1-score   support

           0       0.80      0.83      0.81        99
           1       0.67      0.62      0.64        55

    accuracy                           0.75       154
   macro avg       0.73      0.72      0.73       154
weighted avg       0.75      0.75      0.75       154



/opt/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/opt/venv/lib/python3.13/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
# Oefening 2
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Stap 1 - Dataset inladen
churn = pd.read_csv("Telco-Customer-Churn.csv")

# Stap 2 - Dataset bekijken
print("Eerste en laatste 5 instanties:")
print(churn.head())
print(churn.tail())

print("\nSamenvatting van de dataset:")
print(churn.info())

# Stap 3 - Beoordeling en keuze algoritme
# Deze dataset bestaat uit 20 features, waarvan er 19 bruikbaar zijn (customer_ID niet). Het aantal instanties bedraagt 7043. De dataset is middelgroot, en omdat er zowel numeriek als categorische features aanwezig zijn, lijkt de beste optie een RandomForrest. Een decisiontree heeft te veel risico op overfitting.

# Stap 4 - Preprocessing
# customerID droppen
churn = churn.drop('customerID', axis=1)

# Datatypes corrigeren
churn['TotalCharges'] = pd.to_numeric(churn['TotalCharges'], errors='coerce')
churn['TotalCharges'] = churn['TotalCharges'].fillna(0)

# Features en target splitsen
y = churn['Churn'].map({'Yes': 1, 'No':0})
X = churn.drop('Churn', axis=1)

# Coderen
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
categorical_features = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 
                        'PhoneService', 'MultipleLines', 'InternetService', 
                        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                        'TechSupport', 'StreamingTV', 'StreamingMovies', 
                        'Contract', 'PaperlessBilling', 'PaymentMethod']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Vult de ontstane NaN's in TotalCharges op
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 5. Alles combineren in één grote pipeline inclusief je model
rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', n_estimators=500, oob_score=True, n_jobs=-1, random_state=42))
])

# Train en testset maken
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Stap 5 - Model inladen, trainen en laten voorspellen
rf = rf.fit(X_train, y_train)
y_pred_churn = rf.predict(X_test)

# Stap 6 - Model evalueren en rapporteren
accuracy_churn = accuracy_score(y_test, y_pred_churn)
confusion_churn = confusion_matrix(y_test, y_pred_churn)
report_churn = classification_report(y_test, y_pred_churn)
print(f"\nAccurarcy-score model: {accuracy_churn:.4f}")
print(f"\nConfusion-matrix:\n{confusion_churn}")
print(f"\nClassificatie-rapport:\n{report_churn}")
# Een eerste versie van een Random Forrest presteert over het algemeen redelijk. Door onbalans in de data wordt er echter niet voor beide klassen even nauwkeurig geclassificeerd. Dit kan echter worden opgelost met een simpele hyperparameter die de klassen balanceerd. Het globale model presteert hierdoor niet beter, maar wel constanter over de verschillende klassen.

Eerste en laatste 5 instanties:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV Stre

In [ ]:
# Oefening 3